# The decisive run: is the object-level target measurable at all?

The relatedness test came back null. Before that null means anything, two defects have to be
cleared, because both can manufacture a null on their own.

```
defect 1   ~1 in 3 "objects" were abstract nouns
           activity, walk, swimming, grooming, half, piece, location, environment, species
           nothing to ground, and pmi(dog, walk) is not an object relation

defect 2   -distance from gaze scored 22% against 25% chance -- BELOW chance
           the same geometry beat chance decisively at patch level (22% vs 15.4%)
           when the trusted baseline breaks, suspect the measurement, not the hypothesis
```

## The one idea that settles it: a predictor with known signal

We already measured that objects named in the **answer** take **11x** the damage of objects named
in the question, at matched area. So `is_answer_noun` is a predictor whose signal strength we
already know. It cheats -- it reads the gold answer, so it can never be used at test time -- but
that is exactly what makes it a **ruler for the target**.

```
is_answer_noun ranks above chance   ->  the target IS rankable
                                        -> relatedness failing is a REAL null
is_answer_noun fails too            ->  the target is noise
                                        -> nothing was ever tested; neither the null
                                           nor the 11x can be used for ranking
```

Every previous notebook compared candidate predictors against each other with no way to tell a
dead predictor from a dead target. This one has a thermometer.

## The four changes

1. **Concreteness filter**, calibrated by the model itself against known concrete and known
   abstract words, with both word lists printed so the filter can be checked by eye.
2. **`is_answer_noun` as a validity ruler**, ranked alongside the real predictors.
3. **A damage-spread sweep** instead of one arbitrary cut -- p@1 is plotted against the
   spread threshold, so "it only works on easy examples" is visible rather than hidden.
4. **`N_PER_TYPE = 20`** (was 12). Only 55 of 120 examples survived to the last test; the
   filters here are stricter, so the pool has to be bigger.

## Cost

Grounding ran 2.1 min/100 and ablation 0.7 min/89 last time. At 200 samples budget
**8-12 min** on a T4. Nothing is read from a previous notebook.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy nltk
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, glob, json, time, gc, re, warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import spearmanr, wilcoxon, mannwhitneyu, rankdata
warnings.filterwarnings("ignore")
try:
    from scipy.stats import binomtest as _bt
    def binom_p(k, n, p):
        return _bt(int(k), int(n), p, alternative="greater").pvalue
except ImportError:
    from scipy.stats import binom_test as _bt
    def binom_p(k, n, p):
        return _bt(int(k), int(n), p, alternative="greater")

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS

DATA_ROOT  = "/content/drive/MyDrive/wearvqa_gaze_only"
OUT        = "/content/drive/MyDrive/wearvqa_concrete.pt"
SINKF      = "/content/drive/MyDrive/sink_mask_smolvlm2_n12.pt"
MODEL_ID   = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
N_PER_TYPE = 20
MAX_OBJ, MIN_PATCHES = 8, 2
N_SINK_PROBE = 12
SEED = 0

model, processor, device = S._load_smolvlm(MODEL_ID)
tokenizer = processor.tokenizer

types = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
samples = []
for t in types:
    for jp in sorted(glob.glob(os.path.join(DATA_ROOT, t, "*.json")))[:N_PER_TYPE]:
        m = json.load(open(jp)); ip = jp[:-5] + ".jpg"
        if os.path.exists(ip) and "gaze" in m and m.get("response"):
            samples.append(dict(type=t, img_path=ip, question=m["question"],
                                answer=m["response"], gaze=m["gaze"]))
print(f"{len(types)} types | {len(samples)} samples")

import nltk
NLTK = True
for pkg in ("punkt", "punkt_tab", "averaged_perceptron_tagger",
            "averaged_perceptron_tagger_eng"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
try:
    nltk.pos_tag(nltk.word_tokenize("a red dog runs"))
except Exception as e:
    NLTK = False
    print("nltk POS unavailable:", e)
print("nltk POS tagging:", NLTK)

## 2. Machinery

In [ ]:
def build_inputs(image, question, answer=""):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    return full, int(only["input_ids"].shape[1])

@torch.no_grad()
def answer_logprob(inp, n_prompt, attention_mask=None):
    kw = dict(inp)
    if attention_mask is not None:
        kw["attention_mask"] = attention_mask
    logits = model(**kw).logits[0].float()
    lp = torch.log_softmax(logits[:-1], dim=-1)
    return float(lp.gather(-1, inp["input_ids"][0, 1:, None]).squeeze(-1)[n_prompt-1:].sum())

@torch.no_grad()
def ground(inp):
    patched = S._patch_eager_globals(S._make_raw_capturing_eager(None))
    try:
        model(**inp)
    finally:
        S._unpatch_eager_globals(patched)
    L = int(inp["input_ids"].shape[1])
    acc, n = None, 0
    for m in model.modules():
        p = getattr(m, "_post_attn", None)
        if p is not None and p.shape[-1] == L and p.shape[-2] == L:
            a = p[0].float().mean(0)
            acc = a if acc is None else acc + a; n += 1
        for at in ("_raw_attn_scores", "_post_attn"):
            if hasattr(m, at):
                delattr(m, at)
    return acc / max(n, 1)

def masks_for(ids):
    iid = S._find_image_token_id(model, processor)
    pad = tokenizer.pad_token_id
    im = ids == iid
    tm = (ids != iid) & (ids != (pad if pad is not None else -10**9))
    return torch.nonzero(im).squeeze(-1), torch.nonzero(tm).squeeze(-1)

def token_span(word, pieces):
    text = "".join(pieces).lower()
    hits, start = [], 0
    while True:
        j = text.find(word, start)
        if j < 0:
            break
        hits.append((j, j + len(word))); start = j + 1
    idx, pos = [], 0
    for i, p in enumerate(pieces):
        a, b = pos, pos + len(p)
        if any(b > s and a < e for s, e in hits):
            idx.append(i)
        pos = b
    return idx

# ---------- text-only LM scoring ----------
def _find_text_lm(model):
    for path in ("model.text_model", "model.model.text_model", "model.language_model",
                 "language_model.model", "model.model"):
        m = model
        try:
            for p in path.split("."):
                m = getattr(m, p)
        except AttributeError:
            continue
        if hasattr(m, "embed_tokens") or (isinstance(m, torch.nn.Module) and hasattr(m, "layers")):
            return m
    return None

_TXT = {"mode": None}

@torch.no_grad()
def _text_logits(ids):
    if _TXT["mode"] != "inner":
        try:
            out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            _TXT["mode"] = "top"
            return out.logits[0].float()
        except Exception as e:
            print("top-level text-only forward failed, using inner LM:", type(e).__name__)
            _TXT["mode"] = "inner"; _TXT["lm"] = _find_text_lm(model)
            assert _TXT["lm"] is not None, "could not locate the text decoder"
    h = _TXT["lm"](input_ids=ids, attention_mask=torch.ones_like(ids))
    h = h[0] if isinstance(h, tuple) else h.last_hidden_state
    return model.lm_head(h)[0].float()

@torch.no_grad()
def lm_logprob(prefix, cont):
    n = len(tokenizer(prefix)["input_ids"])
    ids = torch.tensor([tokenizer(prefix + " " + cont)["input_ids"]], device=device)
    lp = torch.log_softmax(_text_logits(ids)[:-1], dim=-1)
    return float(lp.gather(-1, ids[0, 1:, None]).squeeze(-1)[n-1:].sum())

POS_T = "A photograph of a {a}. In the same photograph you can also see a"
NEG_T = "A photograph. In the photograph you can also see a"
_neg_cache, _pmi_cache = {}, {}

def pmi(a, b):
    key = (a, b)
    if key not in _pmi_cache:
        if b not in _neg_cache:
            _neg_cache[b] = lm_logprob(NEG_T, b)
        _pmi_cache[key] = lm_logprob(POS_T.format(a=a), b) - _neg_cache[b]
    return _pmi_cache[key]

_emb_cache = {}

@torch.no_grad()
def emb_vec(w):
    if w not in _emb_cache:
        ids = tokenizer(" " + w, add_special_tokens=False)["input_ids"]
        v = model.get_input_embeddings()(torch.tensor(ids, device=device)).float().mean(0)
        _emb_cache[w] = F.normalize(v, dim=-1).cpu()
    return _emb_cache[w]

def emb_cos(a, b):
    return float(emb_vec(a) @ emb_vec(b))

print("relatedness scorer sanity (must separate):")
for a, b in [("refrigerator","milk"), ("refrigerator","ceiling"),
             ("laptop","keyboard"), ("laptop","banana")]:
    print(f"   pmi({a:<13},{b:<9}) = {pmi(a,b):+.2f}")

## 3. The concreteness filter — calibrated, and printed so it can be checked

`activity`, `walk`, `swimming`, `half`, `location` have nothing to ground and no object relation.
We score every noun by how photographable it is, and set the cut using known-concrete and
known-abstract calibration words rather than a number picked by hand.

```
conc(w) = log P(w | "A photograph of a")  -  log P(w | "The concept of")
```

In [ ]:
CONC_POS, CONC_NEG = "A photograph of a", "The concept of"
_conc_cache = {}

def concreteness(w):
    if w not in _conc_cache:
        _conc_cache[w] = lm_logprob(CONC_POS, w) - lm_logprob(CONC_NEG, w)
    return _conc_cache[w]

CAL_CONCRETE = ["dog","bottle","chair","laptop","tree","shirt","knife","bowl","door","car",
                "cup","table","flower","book","window"]
CAL_ABSTRACT = ["activity","idea","location","amount","concept","purpose","situation",
                "quality","reason","method","process","condition","context","aspect","factor"]

cc = np.array([concreteness(w) for w in CAL_CONCRETE])
ca = np.array([concreteness(w) for w in CAL_ABSTRACT])
THRESH = float((np.median(cc) + np.median(ca)) / 2)
print(f"calibration  concrete median {np.median(cc):+.2f}   abstract median {np.median(ca):+.2f}")
print(f"             overlap: {(ca > np.median(cc)).sum()} abstract above concrete median, "
      f"{(cc < np.median(ca)).sum()} concrete below abstract median")
print(f"THRESHOLD = {THRESH:+.2f}\n")
assert np.median(cc) > np.median(ca), "calibration failed - the probe does not separate"

GENERIC = {"image","picture","photo","thing","things","side","front","back","top","bottom",
           "left","right","part","color","colour","shape","type","kind","number","purpose",
           "way","use","something","area","place"}

def nouns_of(text):
    if NLTK:
        out = [w.lower() for w, t in nltk.pos_tag(nltk.word_tokenize(text))
               if t.startswith("NN") and len(w) > 2]
    else:
        out = [RS._clean_token(w).lower() for w in re.findall(r"[A-Za-z]+", text)
               if len(w) > 2 and not RS.is_stopword_token(w)]
    seen, res = set(), []
    for w in out:
        if w not in seen and w not in GENERIC:
            seen.add(w); res.append(w)
    return res

vocab = sorted({w for s in samples for w in nouns_of(s["question"]) + nouns_of(s["answer"])})
scored = sorted(((concreteness(w), w) for w in vocab), reverse=True)
KEEP = {w for c, w in scored if c >= THRESH}
print(f"corpus nouns {len(vocab)}   kept {len(KEEP)} ({len(KEEP)/len(vocab):.0%})   "
      f"dropped {len(vocab)-len(KEEP)}\n")
kept_words    = [w for c, w in scored if c >= THRESH]
dropped_words = [w for c, w in scored if c <  THRESH][::-1]
print("KEPT  (most concrete first, first 40):")
print("   " + ", ".join(kept_words[:40]))
print("\nDROPPED (least concrete first, first 40):")
print("   " + ", ".join(dropped_words[:40]))
print("\nGATE: the dropped list should be full of abstractions and gerunds, the kept list")
print("full of nameable things. If it is inverted or mixed, stop - the filter is the bug.")

## 4. Sink mask

In [ ]:
if os.path.exists(SINKF):
    b_ = torch.load(SINKF, weights_only=False)
    sinks, sink_score = b_["sink_mask"].bool(), b_["sink_score"]
    print(f"loaded sink mask: {torch.nonzero(sinks).squeeze(-1).tolist()}")
else:
    step = max(1, len(samples) // N_SINK_PROBE)
    sc = []
    for s in samples[::step][:N_SINK_PROBE]:
        o = S.make_smolvlm_output(image=S.load_image(s["img_path"]), question=s["question"])
        if o is None:
            continue
        sc.append(VS.sink_scores(o.post_softmax, o.image_token_mask, o.text_token_mask,
                                 is_post_softmax=True))
        del o; gc.collect(); torch.cuda.empty_cache()
    sink_score = VS.aggregate_sink_scores(sc)
    assert float(sink_score.median()) > 0, "sink scores collapsed to zero"
    sinks = VS.detect_sinks(sink_score)
    torch.save({"model": MODEL_ID, "L_v": sink_score.numel(), "sink_mask": sinks,
                "sink_score": sink_score, "n_probe": len(sc)}, SINKF)
    print(VS.sink_report(sink_score, sinks))

## 5. Ground the concrete nouns only

In [ ]:
objs, t0 = [], time.time()
G = None
n_drop_words = 0
for i, s in enumerate(samples):
    img = S.load_image(s["img_path"])
    inp, n_prompt = build_inputs(img, s["question"], s["answer"])
    ids = inp["input_ids"][0].cpu()
    vpos, tpos = masks_for(ids)
    L_v = len(vpos)
    if G is None:
        G = int(round(math.sqrt(L_v)))

    A = ground(inp)
    pieces = [RS._detok_piece(t) for t in tokenizer.convert_ids_to_tokens(ids[tpos].tolist())]

    qn, an = nouns_of(s["question"]), nouns_of(s["answer"])
    raw = qn + [w for w in an if w not in qn]
    names = [w for w in raw if w in KEEP][:MAX_OBJ]
    n_drop_words += len(raw) - len([w for w in raw if w in KEEP])

    maps, kept, src = [], [], []
    for w in names:
        rows = token_span(w, pieces)
        if not rows:
            continue
        m = A[tpos[rows]][:, vpos].mean(0)
        maps.append(m / m.sum().clamp_min(1e-9)); kept.append(w)
        src.append("Q" if w in qn else "A")

    del A, inp; gc.collect(); torch.cuda.empty_cache()
    if len(maps) < 3:
        objs.append(None); continue

    M = torch.stack(maps, 0)
    lab = M.argmax(0)
    gp = (min(G-1, int(s["gaze"]["y_norm"]*G)) * G + min(G-1, int(s["gaze"]["x_norm"]*G)))
    objs.append(dict(idx=i, names=kept, src=src, M=M, lab=lab, gp=gp,
                     gaze_obj=int(lab[gp]), W=img.size[0], H=img.size[1]))
    if (i+1) % 50 == 0:
        print(f"  grounded {i+1}/{len(samples)}  ({(time.time()-t0)/60:.1f} min)")

good = [o for o in objs if o is not None]
print(f"\nusable {len(good)}/{len(samples)}   objects/image {np.mean([len(o['names']) for o in good]):.1f}"
      f"   abstract words dropped {n_drop_words}")

cors = []
for o in good:
    Mn = F.normalize(o["M"] - o["M"].mean(1, keepdim=True), dim=1)
    C = (Mn @ Mn.T).numpy()
    cors += [C[a, b] for a in range(len(C)) for b in range(a+1, len(C))]
print(f"GATE  mean pairwise map correlation {np.mean(cors):.3f}   (near 1.0 = no discrimination)")
print(f"      gaze object is a QUESTION noun {np.mean([o['src'][o['gaze_obj']]=='Q' for o in good]):.0%}")

## 6. Ablate

In [ ]:
if os.path.exists(OUT):
    res = torch.load(OUT, weights_only=False)
    print(f"resuming from {len(res)}")
else:
    res = []
done = {r["idx"] for r in res}
t0 = time.time()
for o in good:
    if o["idx"] in done:
        continue
    s = samples[o["idx"]]
    inp, n_prompt = build_inputs(S.load_image(s["img_path"]), s["question"], s["answer"])
    ids = inp["input_ids"][0].cpu()
    vpos, _ = masks_for(ids)
    L_v = len(vpos)
    base = answer_logprob(inp, n_prompt)
    n_obj = len(o["names"])
    dmg = torch.full((n_obj,), float("nan")); sz = torch.zeros(n_obj)
    cx = torch.zeros(n_obj); cy = torch.zeros(n_obj)
    for r in range(n_obj):
        sel = ((o["lab"] == r) & (~sinks[:L_v])).nonzero().squeeze(-1)
        sz[r] = len(sel)
        if len(sel) < MIN_PATCHES:
            continue
        cx[r] = float((sel % G).float().mean() + .5) / G
        cy[r] = float((sel // G).float().mean() + .5) / G
        am = inp["attention_mask"].clone(); am[0, vpos[sel]] = 0
        dmg[r] = base - answer_logprob(inp, n_prompt, am)
    res.append(dict(idx=o["idx"], base=base, dmg=dmg, sz=sz, cx=cx, cy=cy,
                    names=o["names"], src=o["src"], gaze_obj=o["gaze_obj"],
                    gp=o["gp"], W=o["W"], H=o["H"], lab=o["lab"]))
    del inp; gc.collect(); torch.cuda.empty_cache()
    if len(res) % 40 == 0:
        torch.save(res, OUT); print(f"  ablated {len(res)}/{len(good)}  ({(time.time()-t0)/60:.1f} min)")
torch.save(res, OUT)
print(f"done in {(time.time()-t0)/60:.1f} min -> {OUT}")

## 7. Build the pairs — now with the validity ruler

`is_answer` reads the gold answer, so it can never run at test time. It is here only as a
**thermometer for the target**: we already know it is worth 11x, so if it cannot rank, nothing can.

In [ ]:
rng = np.random.default_rng(SEED)
shuf = rng.permutation(len(res))
per_ex = []
for e, r in enumerate(res):
    g = r["gaze_obj"]; ga = r["names"][g]
    fr = res[int(shuf[e])]; fake = fr["names"][fr["gaze_obj"]]
    gx = samples[r["idx"]]["gaze"]["x_norm"] * r["W"]
    gy = samples[r["idx"]]["gaze"]["y_norm"] * r["H"]
    loc = []
    for j in range(len(r["names"])):
        if j == g or not math.isfinite(float(r["dmg"][j])):
            continue
        b = r["names"][j]
        dist = math.hypot(float(r["cx"][j])*r["W"] - gx, float(r["cy"][j])*r["H"] - gy)
        loc.append(dict(ex=e, a=ga, b=b, src=r["src"][j],
                        dmg=float(r["dmg"][j]), size=float(r["sz"][j]),
                        pmi=pmi(ga, b), cos=emb_cos(ga, b), pmi_shuf=pmi(fake, b),
                        negdist=-dist, dist=dist,
                        is_answer=1.0 if r["src"][j] == "A" else 0.0))
    if len(loc) >= 3:
        per_ex.append(loc)

rows = [q for p in per_ex for q in p]
D = {k: np.array([x[k] for x in rows], float)
     for k in ("dmg","size","pmi","cos","pmi_shuf","negdist","dist","is_answer")}
SRC = np.array([x["src"] for x in rows])
print(f"{len(rows)} pairs from {len(per_ex)} examples   "
      f"mean others/example {np.mean([len(p) for p in per_ex]):.1f}")

qd, ad = D["dmg"][SRC=="Q"], D["dmg"][SRC=="A"]
print(f"\nthe 11x, re-measured on concrete nouns only:")
print(f"  QUESTION objects   damage {qd.mean():.3f}   size {D['size'][SRC=='Q'].mean():.1f} patches  n={len(qd)}")
print(f"  ANSWER-only objs   damage {ad.mean():.3f}   size {D['size'][SRC=='A'].mean():.1f} patches  n={len(ad)}")
if len(qd) > 5 and len(ad) > 5:
    print(f"  Mann-Whitney (answer greater) p {mannwhitneyu(ad, qd, alternative='greater')[1]:.3g}")
print(f"\n  relatedness vs is_answer : rho {spearmanr(D['pmi'], D['is_answer'])[0]:+.3f}"
      "     <- if ~0, relatedness is blind to the only thing that predicts damage")

## 8. THE DECISIVE SWEEP

p@1 against the damage-spread threshold. Ties get fractional credit, so the binary `is_answer`
predictor is not penalised for tying.

In [ ]:
PREDS = [("is_answer  [RULER, cheats]", "is_answer"),
         ("relatedness (LM-PMI)",       "pmi"),
         ("relatedness (emb cos)",      "cos"),
         ("-distance from gaze",        "negdist"),
         ("CTRL shuffled PMI",          "pmi_shuf")]
SPREADS = [0.0, 0.05, 0.1, 0.2, 0.4, 0.8]

def prec_at_1(key, pairs):
    hit = 0.0
    for p in pairs:
        x = np.array([q[key] for q in p]); y = np.array([q["dmg"] for q in p])
        top = np.flatnonzero(x >= x.max() - 1e-12)
        hit += (1.0/len(top)) if int(y.argmax()) in top else 0.0
    return hit / len(pairs)

def mean_rho(key, pairs):
    o = []
    for p in pairs:
        x = np.array([q[key] for q in p], float); y = np.array([q["dmg"] for q in p], float)
        if len(set(x.tolist())) > 1 and len(set(y.tolist())) > 1:
            v = spearmanr(x, y)[0]
            if np.isfinite(v):
                o.append(v)
    return np.array(o)

curves = {k: [] for _, k in PREDS}
print(f"{'spread':>7} {'n_ex':>5} {'chance':>7}  " +
      "  ".join(f"{l.split('(')[0].split('[')[0].strip()[:13]:>13}" for l, _ in PREDS))
print("-" * 100)
for th in SPREADS:
    sub = [p for p in per_ex
           if (max(q["dmg"] for q in p) - min(q["dmg"] for q in p)) > th and len(p) >= 3]
    if len(sub) < 10:
        print(f"{th:>7.2f} {len(sub):>5}   (too few examples, stopping the sweep)"); break
    ch = np.mean([1/len(p) for p in sub])
    line = f"{th:>7.2f} {len(sub):>5} {ch:>7.0%}  "
    for _, k in PREDS:
        v = prec_at_1(k, sub); curves[k].append((th, len(sub), v, ch))
        line += f"  {v:>12.0%}"
    print(line)
print("-" * 100)
print("read DOWN the is_answer column first. If it never rises clearly above chance,")
print("the target is not rankable and no other column means anything.")

In [ ]:
th0 = SPREADS[0]
sub0 = [p for p in per_ex if len(p) >= 3]
print(f"full set, n={len(sub0)} examples\n")
print(f"{'predictor':<28} {'per-ex rho':>11} {'p vs 0':>9} {'p@1':>7} {'p@1 vs chance':>15}")
print("-" * 76)
ch0 = np.mean([1/len(p) for p in sub0])
store = {}
for label, k in PREDS:
    r = mean_rho(k, sub0); store[k] = r
    p0 = wilcoxon(r)[1] if len(r) > 5 and np.any(r != 0) else float("nan")
    pk = prec_at_1(k, sub0)
    pb = binom_p(round(pk*len(sub0)), len(sub0), ch0)
    print(f"{label:<28} {r.mean():>+11.3f} {p0:>9.3g} {pk:>7.0%} {pb:>15.3g}")
print("-" * 76)
print(f"chance {ch0:.0%}")

print("\n=== head to head (paired, within example) ===")
def h2h(k1, k2, l1, l2):
    n = min(len(store[k1]), len(store[k2]))
    a, b = store[k1][:n], store[k2][:n]
    p = wilcoxon(a, b)[1] if np.any(a != b) else float("nan")
    print(f"  {l1} {a.mean():+.3f}  vs  {l2} {b.mean():+.3f}   "
          f"diff {(a-b).mean():+.3f}  p {p:.3g}  {l1} wins {(a>b).mean():.0%}")
h2h("pmi", "negdist", "relatedness", "-distance  ")
h2h("pmi", "pmi_shuf", "relatedness", "shuffled   ")
h2h("is_answer", "pmi", "is_answer  ", "relatedness")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
cols = {"is_answer":"tab:purple","pmi":"tab:green","cos":"tab:olive",
        "negdist":"tab:red","pmi_shuf":"lightgray"}
for label, k in PREDS:
    if not curves[k]:
        continue
    xs = [c[0] for c in curves[k]]; ys = [c[2] for c in curves[k]]
    ax[0].plot(xs, ys, "o-", label=label, color=cols[k],
               lw=2.5 if k == "is_answer" else 1.5)
ax[0].plot([c[0] for c in curves["pmi"]], [c[3] for c in curves["pmi"]],
           "k--", label="chance")
ax[0].set_xlabel("damage-spread threshold (nats)"); ax[0].set_ylabel("precision@1")
ax[0].set_title("does anything rank once the flat examples are removed?")
ax[0].legend(fontsize=8)

ax[1].bar(range(len(PREDS)), [store[k].mean() for _, k in PREDS],
          color=[cols[k] for _, k in PREDS])
ax[1].axhline(0, c="k", lw=.8)
ax[1].set_xticks(range(len(PREDS)))
ax[1].set_xticklabels([l.split("(")[0].split("[")[0].strip() for l, _ in PREDS],
                      rotation=20, ha="right", fontsize=8)
ax[1].set_ylabel("mean within-example Spearman vs damage")
ax[1].set_title("full set")
plt.tight_layout(); plt.show()

## 9. The verdict, computed

In [ ]:
best_ruler = max(c[2] - c[3] for c in curves["is_answer"])
ruler_at = [c for c in curves["is_answer"] if c[2] - c[3] == best_ruler][0]
n_at = ruler_at[1]
ruler_p = binom_p(round(ruler_at[2]*n_at), n_at, ruler_at[3])
rel = store["pmi"]; shf = store["pmi_shuf"]; dst = store["negdist"]
n = min(len(rel), len(shf))
p_shuf = wilcoxon(rel[:n], shf[:n])[1] if np.any(rel[:n] != shf[:n]) else 1.0
n2 = min(len(rel), len(dst))
p_dist = wilcoxon(rel[:n2], dst[:n2])[1] if np.any(rel[:n2] != dst[:n2]) else 1.0

rel_wins = p_shuf < 0.05 and p_dist < 0.05 and rel[:n].mean() > shf[:n].mean() \
           and rel[:n2].mean() > dst[:n2].mean()

print(f"RULER   is_answer best margin over chance {best_ruler:+.0%} "
      f"at spread>{ruler_at[0]:.2f} (n={n_at}), binomial p {ruler_p:.3g}")
print(f"BELIEF  relatedness vs shuffled   p {p_shuf:.3g}   wins {(rel[:n]>shf[:n]).mean():.0%}")
print(f"        relatedness vs -distance  p {p_dist:.3g}   wins {(rel[:n2]>dst[:n2]).mean():.0%}\n")

# order matters: relatedness winning IS itself proof the target is rankable, so that
# branch has to be tested before the ruler can veto anything.
if rel_wins:
    print("VERDICT  THE INTUITION IS SUPPORTED.")
    print("  Relatedness beats both its own shuffled control and the geometry baseline, which")
    print("  by itself proves the target is rankable - the ruler cannot veto this.")
    print("  Rebuild FRM on language-space relatedness rather than a learned dot product, and")
    print("  the grid/segmenter work is now worth the compute.")
elif ruler_p > 0.05:
    print("VERDICT  THE TARGET IS NOT RANKABLE.")
    print("  Nothing ranks - not even a predictor we measured at 11x. The object-level ground")
    print("  truth is noise, so the relatedness null is NOT evidence against the intuition;")
    print("  nothing was tested.")
    print("  Next: do_image_splitting=True (81 -> 324+ tokens) and a real segmenter, so that")
    print("  an object is more than a handful of 80x142 px patches. That is the 2-3 hour path.")
else:
    print("VERDICT  A REAL NULL.")
    print("  The target IS rankable - is_answer proves it - and relatedness still fails to")
    print("  beat its shuffled control. So relatedness to the gaze object genuinely does not")
    print("  predict what the answer needs. The thread closes.")
    print("  What survives: the answer text names what matters (the 11x), which is free")
    print("  supervision at TRAINING time even though it is unavailable at test time.")

## 10. How to read this

Three outcomes, and the notebook prints which one it landed on.

| `is_answer` ruler | relatedness | verdict |
|---|---|---|
| fails to beat chance | anything | **nothing was tested.** The object-level target is noise. The earlier null is void, and so is using the 11x for ranking. Go to `do_image_splitting=True` + a segmenter. |
| beats chance | ties its shuffle | **a real null.** Relatedness to the gaze object does not predict what the answer needs. Close the thread; keep the 11x as a training-time signal. |
| beats chance | beats shuffle **and** distance | **supported.** Semantics beats geometry. Rebuild FRM on relatedness. |

Two gates come first, as always: the concreteness lists in section 3 must look right by eye, and
the map-correlation gate in section 5 must stay well below 0.9. The last ten notebooks were lost
to reading results off a broken measurement — the ruler in section 8 exists so that cannot happen
quietly again.